In [6]:
from openai import OpenAI

# Local Mac — LM Studio
local = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="not-needed",
)

# GPU cluster
cluster = OpenAI(
    base_url="http://137.226.117.70:8000/v1",
    api_key="not-needed",
)

print("LM Studio models:")
for m in local.models.list():
    print(" ", m.id)

print("\nvLLM models:")
for m in cluster.models.list():
    print(" ", m.id)

LM Studio models:
  qwen3-8b-mlx
  meta/llama-3.3-70b
  google/gemma-4-e4b
  text-embedding-nomic-embed-text-v1.5

vLLM models:
  Qwen/Qwen3-8B


In [8]:
import time

PROMPT = "Comprehensively explain me process mining."

# Replace these with the exact IDs printed by Cell 1
LOCAL_MODEL = "qwen/qwen3-8b"
CLUSTER_MODEL = "Qwen/Qwen3-8B"


def benchmark(client, model):
    start = time.perf_counter()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": PROMPT}
        ],
        temperature=0,
        max_tokens=2000,
    )

    elapsed = time.perf_counter() - start

    message = response.choices[0].message
    usage = response.usage

    return {
        "elapsed_s": elapsed,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "tokens_per_second": usage.completion_tokens / elapsed,
        "finish_reason": response.choices[0].finish_reason,
        "content": message.content,
        "reasoning": getattr(message, "reasoning_content", None),
    }


local_result = benchmark(local, LOCAL_MODEL)
cluster_result = benchmark(cluster, CLUSTER_MODEL)


for name, r in [
    ("M4 Pro / LM Studio", local_result),
    ("GPU cluster / vLLM", cluster_result),
]:
    print(f"\n{'=' * 60}")
    print(name)
    print("=" * 60)

    print(f"Time:              {r['elapsed_s']:.2f} s")
    print(f"Prompt tokens:     {r['prompt_tokens']}")
    print(f"Completion tokens: {r['completion_tokens']}")
    print(f"Approx tokens/sec: {r['tokens_per_second']:.1f}")
    print(f"Finish reason:     {r['finish_reason']}")

    if r["reasoning"]:
        print("\n--- Reasoning ---")
        print(r["reasoning"])

    print("\n--- Answer ---")
    print(r["content"])


M4 Pro / LM Studio
Time:              118.36 s
Prompt tokens:     16
Completion tokens: 1797
Approx tokens/sec: 15.2
Finish reason:     stop

--- Reasoning ---

Okay, the user wants a comprehensive explanation of process mining. Let me start by recalling what I know about it. Process mining is a technique that uses event logs from IT systems to analyze business processes. But I need to make sure I cover all the key aspects.

First, I should define process mining clearly. Maybe start with its purpose: to discover, monitor, and improve processes by analyzing event data. Then break it down into the three main activities: discovery, conformance checking, and enhancement. That's a common structure I've seen in literature.

Wait, the user might not be familiar with event logs. I should explain what those are—like records of activities, timestamps, and case IDs. Also, mention the types of data sources, such as ERP systems or BPMN models.

I need to explain each of the three main areas in det